<a href="https://colab.research.google.com/github/hiba-ai-ing/projet-mL/blob/main/SABIR_HIBA_TP4_ML_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Travaux Pratiques N°4 : Comparaison des modèles de
régression non linéaire (polynomiale et SVR) par
validation croisée et GridSearchCV.


**Partie I : Questions de réflexion**

 Le SVR cherche à atteindre deux objectifs simultanément pour déterminer sa fonction de
prédiction:

1 er objectif:
**une fonction peu inclinée:**
 parce que plus la pente est petite , moins la fonction est sensible aux petits variations et robuste au bruit

 2 eme objectif :**Pénaliser uniquement les dépassements hors du tube** ou les erreurs dans le tube sont tolérée , et ehors du tube les erreurs sont pénalisées.

 -il est nécessaire e trouver un compromis entre eux par l'hyperparametre C , en valeur importante C pénalise fortement les dépassement et si c 'est petit  le C  prévilége la platitude , c'est pour ça il est nécessaie de trouver un compromis entre les deux pour réaliser les 2 objectifs.

 **Question 2 :expliquer  la signification de chaque hyperparamètre du SVR
et son influence sur le comportement du modèle.**

**C**:hyperparametre  qui  contribue  dans le probleme d'optimisattion du SVR , ou il trouve un compromis entre  la platitude et la pénalisation  des dépasseements hors tube .

**epsilon (ε):**  permet de cadrer la zone de tolérance où les dépassement et les résidus sont tolérée , et le cout d'erreur  est calculé seulement pour les dépassement des supports vecteurs .

kernel:Définit le type de fonction utilisé pour modéliser les relations entre les explicatives, dans le cours il a pris la valeur rbf , c'une relation non linéaire plus flexible.
 Le choix du noyau influence la forme de la fonction prédite par le SVR.

**Question3)** la standardisation est nécessaire pour la variable explicatifs  parce que le noyau rbf repose sur les distances entres observations si les variables ne sont pas sur meme echelle , la proximité est faussée.

**Question 4)** il est utile de standariser y avant l'entrainement du SVR  , et puisque la standardisation modifie l’échelle des erreurs de prédiction. Il peut donc être nécessaire de réajuster C pour trouver le meilleur compromis entre la tolérance aux erreurs et la complexité du modèle.

et pour epsilon : après standardisation, epsilon s’exprime dans l’échelle standardisée de \(y\), et non plus dans son unité d’origine. Il devient donc plus facile de choisir une marge de tolérance adaptée

lors de l'interprétation des prédictions obtenues  il faut se rappeler de faire la transformation inverse pour retrouvé les valeurs dans l'unité d'origine de y.

**Question 5):**

 le paramètre gamma (γ) contrôle l’influence des
vecteurs de support
K(xi,x)=exp(-gamma llx-xi1||2 ) donc pout gamma petit : la similarité diminue lentement  et parsuite  même les points assez loin gardent de l'influence  on obtiendra une courbe plus lisse.
 et si  gamma est grand : la similarité diminue vite donc seuls les points très proches influencent la prédiction

ce que signifie
gamma='scale' dans Scikit-learnce
signifie que la valeur de gamma est calculée automatiquement à partir des données d’entrée X, selon la formule cité dans le cours

cette option adapte automatiquement gamma à l’échelle et à la variance des variables, sans avoir à fixer manuellement sa valeur.



Influence : gamma contrôle la portée d’influence des observations dans le noyau RBF. Une petite valeur produit généralement une courbe plus lisse, tandis qu’une grande valeur permet une courbe plus flexible.


**Partie II : Étude de cas : Prédiction de la surface moyenne de lésions tumorales.**



In [13]:
# Q1) Bibliothèques nécessaires
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn. linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Valeur de référence pour les decoupages aleatoires
SEED = 42
np.random. seed (SEED)

**kfold**:permet de  diviser les données en k sous ensembles pour effectur une validation croisé a chaque itération k-1 fold pour l'entrainement et 1 fold pour la validation , le nombre d'itération est k .
et en fin une autre partie non encore touché est utilisée pour le test .

**cross_validate:** permet d’évaluer les performances d’un modèle par validation croisée. Il entraîne et évalue le modèle plusieurs fois, puis retourne les scores obtenus pour chaque pli, selon les métriques demandées.

**GridSearchCV:** permet de rechercher automatiquement les meilleurs hyperparamètres d’un modèle en testant plusieurs combinaisons. Pour chaque combinaison, il utilise la validation croisée afin de comparer les performances et sélectionner la meilleure configuration

In [14]:
# Q2) Chargement du jeu de données
donnees = load_breast_cancer(as_frame=True) . frame
# Les 7 variables explicatives conservées
colonnes_x = ["mean radius", "mean texture", "mean perimeter",
"mean smoothness", "mean compactness",
"mean concavity", "mean symmetry"]
X = donnees [colonnes_x]
y = donnees["mean area"]
print("Dimensions de x :", X. shape)
print("Dimensions de y :", y. shape)
print(pd.concat([X, y], axis=1).head())

Dimensions de x : (569, 7)
Dimensions de y : (569,)
   mean radius  mean texture  mean perimeter  mean smoothness  \
0        17.99         10.38          122.80          0.11840   
1        20.57         17.77          132.90          0.08474   
2        19.69         21.25          130.00          0.10960   
3        11.42         20.38           77.58          0.14250   
4        20.29         14.34          135.10          0.10030   

   mean compactness  mean concavity  mean symmetry  mean area  
0           0.27760          0.3001         0.2419     1001.0  
1           0.07864          0.0869         0.1812     1326.0  
2           0.15990          0.1974         0.2069     1203.0  
3           0.28390          0.2414         0.2597      386.1  
4           0.13280          0.1980         0.1809     1297.0  


In [15]:
#Q3) Partitionnement et validation croisee
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.20, random_state=SEED)
# Cinq splis avec melange reproductible des observations
CV = KFold(n_splits=5, shuffle=True, random_state=SEED)
# À cette etape, nous preparons uniquement la strategie de validation croisee.
# Nous ne lui fournissons pas encore x_train ni y_train.
print("x_train :", X_train.shape)
print("x_test :", X_test. shape)
# Trois criteres calcules a chaque split de validation
metriques = {
"R2": "r2",
"MAE": "neg_mean_absolute_error",
"RMSE": "neg_root_mean_squared_error"

}

x_train : (455, 7)
x_test : (114, 7)


-le nombre d'observations conservé à l'entrainement est 455 observations , alors que le nombre d'observations  en test est 114

le test ne dois pas intervenir dans  la recherche des hyperparametres   pour ne pas  causer le data leakage /la fuite de données .

In [16]:
# Q4) Comparaison guidée des degrés polynomiaux
# Tableau des resultats par degre polynomial
resultats_poly = []
for degre in [2, 3, 4, 5]:
   # Transformer les 7 variables en caracteristiques polynomiales
   poly = PolynomialFeatures(degree=degre, include_bias=False)
   x_train_poly = poly.fit_transform(X_train)
   # Évaluer le degre par validation croisee a 5 splits
   scores = cross_validate(
        LinearRegression(), x_train_poly, y_train, cv=CV , scoring=metriques)
   # Conserver la moyenne des metriques obtenues sur les cinq splits
   resultats_poly.append({
           "Degré": degre,
           "Nb_caracteristiques": x_train_poly. shape[1],
           "MAE_CV": -scores["test_MAE"].mean(),
           "RMSE_CV": -scores["test_RMSE"].mean(),
           "R2_CV": scores["test_R2"].mean()})
resultats_poly = pd.DataFrame(resultats_poly)
print(resultats_poly.round(4).to_string(index=False))
# Choisir le degre qui maximise le R2 moyen
ligne_poly = resultats_poly.loc[resultats_poly["R2_CV"].idxmax()]
meilleur_degre = int(ligne_poly["Degré"])
print("Meilleur degre polynomial :", meilleur_degre)

 Degré  Nb_caracteristiques  MAE_CV  RMSE_CV  R2_CV
     2                   35  6.8254  13.8520 0.9984
     3                  119 10.4212  22.7785 0.9955
     4                  329 36.3593 122.4601 0.8779
     5                  791 49.8835 209.5537 0.5354
Meilleur degre polynomial : 2


**Réponse Q4)**

Le meilleur degré polynomial est le degré 2, car il présente les meilleures performances selon les trois métriques : la MAE et la RMSE sont les plus faibles (respectivement 6,8254 et 13,8520), tandis que le coefficient \(R^2\) est le plus élevé (0,9984).
Lorsque le degré augmente de 2 à 5, le nombre de caractéristiques augmente fortement, passant de 35 à 791. Cette augmentation permet au modèle de représenter des relations plus complexes, mais elle peut aussi entraîner un surapprentissage et une moins bonne généralisation.
On observe que la MAE et la RMSE augmentent considérablement, tandis que \(R^2\) diminue de 0,9984 à 0,5354. Cela indique une dégradation progressive des performances du modèle lorsque le degré augmente. Le degré 2 offre donc le meilleur compromis entre complexité et précision, tandis que les degrés 4 et 5 semblent trop complexes pour ces données

In [9]:
# Q5) Préparation du SVR avec standardisation intégrée
from sklearn.pipeline import make_pipeline
modele_svr = make_pipeline(StandardScaler(),
                           SVR(kernel='rbf') )

StandardScaler() : standardise les variables explicatives \(X\) pour qu’elles soient comparables en échelle.



SVR(kernel='rbf') : construit un modèle SVR utilisant le noyau RBF pour apprendre des relations non linéaires.



make_pipeline() : enchaîne automatiquement les deux étapes. Lors de la validation croisée, le scaler est ajusté uniquement sur les données d’entraînement de chaque split, ce qui évite les fuites d’information.

In [10]:
# Q6) Recherche des hyperparamètres du SVR avec GridSearchCV
# Recherche des 36 combinaisons du noyau RBF par 5 splits

recherche_svr = GridSearchCV(
    estimator=modele_svr,
    param_grid={
        "svr__C": [10, 100, 1000],
        "svr__epsilon": [1, 10, 30],
        "svr__gamma": ["scale", 0.01, 0.1, 1]
    },
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    refit="R2",
    cv=5,
    n_jobs=-1
)

recherche_svr.fit(X_train, y_train)

resultats_svr = pd.DataFrame(recherche_svr.cv_results_)

resultats_svr["MAE_CV"] = -resultats_svr["mean_test_MAE"]
resultats_svr["RMSE_CV"] = -resultats_svr["mean_test_RMSE"]
resultats_svr["R2_CV"] = resultats_svr["mean_test_R2"]

colonnes = [
    "param_svr__C",
    "param_svr__epsilon",
    "param_svr__gamma",
    "MAE_CV",
    "RMSE_CV",
    "R2_CV"
]

print("Les 5 meilleures configurations :")
print(
    resultats_svr[colonnes]
    .sort_values("R2_CV", ascending=False)
    .head(5)
    .round(4)
    .to_string(index=False)
)

print("Meilleurs paramètres :", recherche_svr.best_params_)
print("Meilleur R2 moyen CV :", round(recherche_svr.best_score_, 4))

Les 5 meilleures configurations :
 param_svr__C  param_svr__epsilon param_svr__gamma  MAE_CV  RMSE_CV  R2_CV
         1000                   1             0.01  9.9771  25.5295 0.9940
         1000                  10             0.01 11.7393  27.5057 0.9931
         1000                  30             0.01 15.9005  29.7617 0.9922
         1000                   1              0.1 15.4751  42.1132 0.9830
         1000                  10              0.1 18.1772  44.6269 0.9817
Meilleurs paramètres : {'svr__C': 1000, 'svr__epsilon': 1, 'svr__gamma': 0.01}
Meilleur R2 moyen CV : 0.994


Réponse Q6)
36 configurations sont testées : \(3 * 3 * 4 = 36).


180 entraînements et évaluations sont réalisés : (36 * 5 itérations = 180).



refit="R2" sélectionne la combinaison qui obtient le meilleur (R^2) moyen en validation croisée, puis réentraîne le meilleur modèle sur toutes les données d’entraînement.

In [11]:
# Q7) Analyse des vecteurs de support

# Recuperation du meilleur SVR obtenu dans Q6
meilleur_svr = recherche_svr.best_estimator_.named_steps["svr"]

# Indices des vecteurs de support
indices_sv = meilleur_svr.support_

# Affichage des indices
print("Indices des vecteurs de support :")
print(indices_sv)

# Nombre de vecteurs de support
nb_sv = len(indices_sv)
print("Nombre de vecteurs de support :", nb_sv)

# Pourcentage de vecteurs de support
pourcentage_sv = nb_sv / len(X_train) * 100
print("Pourcentage de vecteurs de support :",
      round(pourcentage_sv, 2))

# Affichage des 10 premieres observations d'origine
print("Les 10 premieres observations devenues vecteurs de support :")
print(X_train.iloc[indices_sv[:10]])

Indices des vecteurs de support :
[  0   1   2   4   5   6   9  12  14  15  16  17  18  19  20  21  24  25
  27  28  29  30  31  32  33  34  35  36  40  41  42  43  44  45  46  47
  48  50  51  52  53  54  55  58  59  62  63  64  65  66  67  68  69  71
  72  74  75  76  80  82  84  86  87  88  90  91  92  94  95  96  97  98
 100 101 103 104 105 107 108 109 110 111 112 113 114 115 116 117 118 119
 120 121 122 124 125 126 128 130 131 132 137 138 139 140 141 144 145 146
 147 148 149 150 151 152 154 155 156 158 159 160 162 163 164 165 166 167
 168 169 170 171 172 173 175 176 177 179 180 181 182 183 185 186 187 188
 189 190 191 192 193 195 196 197 198 199 200 201 203 204 206 207 208 209
 211 212 213 215 217 218 219 220 221 222 223 225 226 227 228 229 231 234
 235 236 237 238 239 240 242 243 244 245 246 247 248 249 250 251 252 253
 254 255 256 257 258 259 262 263 264 265 268 270 272 273 274 276 277 278
 279 280 282 284 285 286 287 288 289 290 291 292 293 294 296 297 298 299
 303 304 305 306 

. Explication du rôle des vecteurs de support
Les vecteurs de support sont les observations d'entraînement qui contribuent directement à la construction de la fonction de prédiction du modèle SVR.
- Les indices permettent d'identifier les positions des observations qui sont devenues des vecteurs de support.
- Le nombre indique combien d'observations sont utilisées comme vecteurs de support.
- Le pourcentage représente leur proportion parmi toutes les observations d'entraînement.
- Les dix premières observations permettent d'examiner les valeurs d'origine de quelques vecteurs de support.
Dans un SVR utilisant le noyau RBF, ces observations contribuent à la prédiction grâce à leurs similarités avec les nouvelles observations. Les observations qui ne sont pas des vecteurs de support n'ont pas de contribution directe à la fonction de prédiction finale.
Interprétation de vos résultats : vous avez obtenu 363 vecteurs de support, soit 79,78 % des observations d'entraînement. Cela signifie que le modèle s'appuie sur une grande partie des données d'entraînement pour effectuer ses prédictions. Ce pourcentage élevé ne prouve pas, à lui seul, que le modèle est en surapprentissage.

**Q8 — Comparaison des meilleurs modèles**
Modèle	MAE moyen (CV)	RMSE moyen (CV)	R² moyen (CV)
Régression polynomiale (degré 2)	6,8254	13,8520	0,9984
SVR (C = 1000, ε = 1, γ = 0,01)	9,9771	25,5295	0,9940
Bilan comparatif et conclusion
Les résultats montrent que la régression polynomiale de degré 2 est plus performante que le SVR. En effet, elle obtient une MAE de 6,8254 et une RMSE de 13,8520, inférieures à celles du SVR, qui sont respectivement de 9,9771 et 25,5295. De plus, son coefficient \(R^2\) de 0,9984 est supérieur à celui du SVR, qui est de 0,9940.
La régression polynomiale de degré 2 est donc le meilleur modèle selon les trois métriques, car elle présente des erreurs de prédiction plus faibles et un pouvoir explicatif plus élevé.
Si les métriques ne conduisent pas au même choix, le critère principal doit être fixé à l’avance en fonction de l’objectif du problème. Par exemple, la RMSE peut être privilégiée si les grandes erreurs de prédiction sont particulièrement pénalisantes. Le modèle doit ensuite être choisi selon ce critère, sans modifier celui-ci après avoir observé les résultats.

In [18]:
# Q9) Entraînement final et évaluation sur Test

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Utiliser le meilleur degré trouvé en Q4
poly = PolynomialFeatures(degree=meilleur_degre, include_bias=False)

# Transformer Train et Test
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Entraîner le modèle sur tout Train
modele_poly_final = LinearRegression()
modele_poly_final.fit(X_train_poly, y_train)

# Prédire sur Test
y_pred_test = modele_poly_final.predict(X_test_poly)

# Calculer les métriques sur Test
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test = r2_score(y_test, y_pred_test)

# Calculer R² sur Train
y_pred_train = modele_poly_final.predict(X_train_poly)
r2_train = r2_score(y_train, y_pred_train)

# Afficher les résultats
print("Meilleur degré :", meilleur_degre)
print("MAE sur Test :", round(mae_test, 4))
print("RMSE sur Test :", round(rmse_test, 4))
print("R² sur Test :", round(r2_test, 4))
print("R² sur Train :", round(r2_train, 4))

Meilleur degré : 2
MAE sur Test : 6.424
RMSE sur Test : 11.5047
R² sur Test : 0.9989
R² sur Train : 0.9992



2.
Le modèle de régression polynomiale de degré 2 a été entraîné sur l'ensemble du jeu Train, puis évalué sur le jeu Test. Il obtient une MAE de 6,4240, une RMSE de 11,5047 et un \(R^2\) de 0,9989. Ces résultats montrent que le modèle réalise des prédictions très précises sur les données de test.
Les performances sur Test sont proches de celles obtenues en validation croisée. De plus, le \(R^2\) sur Train est de 0,9992, légèrement supérieur au \(R^2\) sur Test. Cet écart est faible, ce qui indique que le modèle généralise bien aux nouvelles données et ne présente pas de signe important de surapprentissage.
En conclusion, la régression polynomiale de degré 2 est un modèle adapté à ce jeu de données, car elle conserve d'excellentes performances lors de l'évaluation finale sur Test.

In [20]:
# Q10) Stabilité face aux variations du découpage

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

seeds = [42, 7, 21, 123,1,300]
resultats = []

for SEED in seeds:

    print("\n========== SEED =", SEED, "==========")

    # 1. Séparer les données
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED
    )

    # Validation croisée à 5 splits
    CV = KFold(n_splits=5, shuffle=True, random_state=SEED)

    # 2. Chercher le meilleur degré polynomial
    resultats_poly = []

    for degre in [2, 3, 4, 5]:

        poly = PolynomialFeatures(degree=degre, include_bias=False)
        X_train_poly = poly.fit_transform(X_train)

        scores = cross_validate(
            LinearRegression(),
            X_train_poly,
            y_train,
            cv=CV,
            scoring={
                "MAE": "neg_mean_absolute_error",
                "RMSE": "neg_root_mean_squared_error",
                "R2": "r2"
            }
        )

        resultats_poly.append({
            "Degré": degre,
            "MAE_CV": -scores["test_MAE"].mean(),
            "RMSE_CV": -scores["test_RMSE"].mean(),
            "R2_CV": scores["test_R2"].mean()
        })

    tableau_poly = pd.DataFrame(resultats_poly)
    ligne_poly = tableau_poly.loc[tableau_poly["R2_CV"].idxmax()]
    meilleur_degre = int(ligne_poly["Degré"])

    # 3. Chercher les meilleurs paramètres SVR
    modele_svr = make_pipeline(
        StandardScaler(),
        SVR(kernel="rbf")
    )

    grille = {
        "svr__C": [10, 100, 1000],
        "svr__epsilon": [1, 10, 30],
        "svr__gamma": ["scale", 0.01, 0.1, 1]
    }

    recherche_svr = GridSearchCV(
        modele_svr,
        grille,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },
        refit="R2",
        cv=CV,
        n_jobs=-1
    )

    recherche_svr.fit(X_train, y_train)

    # Résultats CV du meilleur SVR
    i = recherche_svr.best_index_
    mae_svr_cv = -recherche_svr.cv_results_["mean_test_MAE"][i]
    rmse_svr_cv = -recherche_svr.cv_results_["mean_test_RMSE"][i]
    r2_svr_cv = recherche_svr.cv_results_["mean_test_R2"][i]

    # 4. Comparer les deux modèles selon le R² moyen CV
    if ligne_poly["R2_CV"] >= r2_svr_cv:
        modele_retenu = "Polynomiale"
        poly_final = PolynomialFeatures(
            degree=meilleur_degre, include_bias=False
        )
        X_train_final = poly_final.fit_transform(X_train)
        X_test_final = poly_final.transform(X_test)

        modele_final = LinearRegression()
        modele_final.fit(X_train_final, y_train)

        pred_test = modele_final.predict(X_test_final)
        pred_train = modele_final.predict(X_train_final)

    else:
        modele_retenu = "SVR"
        modele_final = recherche_svr.best_estimator_
        pred_test = modele_final.predict(X_test)
        pred_train = modele_final.predict(X_train)

    # 5. Évaluer le modèle retenu sur Test et calculer R² sur Train
    mae_test = mean_absolute_error(y_test, pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
    r2_test = r2_score(y_test, pred_test)
    r2_train = r2_score(y_train, pred_train)

    # 6. Afficher les résultats
    print("Meilleur degré polynomial :", meilleur_degre)
    print("Meilleurs paramètres SVR :", recherche_svr.best_params_)
    print("Modèle retenu :", modele_retenu)
    print("MAE sur Test :", round(mae_test, 4))
    print("RMSE sur Test :", round(rmse_test, 4))
    print("R² sur Test :", round(r2_test, 4))
    print("R² sur Train :", round(r2_train, 4))

    resultats.append({
        "Seed": SEED,
        "Meilleur degré": meilleur_degre,
        "Paramètres SVR": recherche_svr.best_params_,
        "Modèle retenu": modele_retenu,
        "MAE Test": mae_test,
        "RMSE Test": rmse_test,
        "R² Test": r2_test
    })

# Tableau récapitulatif
tableau_final = pd.DataFrame(resultats)

print("\n========== TABLEAU RÉCAPITULATIF ==========")
print(tableau_final.round(4).to_string(index=False))


========== SEED = 42 ==========
Meilleur degré polynomial : 2
Meilleurs paramètres SVR : {'svr__C': 1000, 'svr__epsilon': 1, 'svr__gamma': 0.01}
Modèle retenu : Polynomiale
MAE sur Test : 6.424
RMSE sur Test : 11.5047
R² sur Test : 0.9989
R² sur Train : 0.9992

========== SEED = 7 ==========
Meilleur degré polynomial : 2
Meilleurs paramètres SVR : {'svr__C': 1000, 'svr__epsilon': 1, 'svr__gamma': 0.01}
Modèle retenu : Polynomiale
MAE sur Test : 5.9206
RMSE sur Test : 10.8626
R² sur Test : 0.9988
R² sur Train : 0.9992

========== SEED = 21 ==========
Meilleur degré polynomial : 2
Meilleurs paramètres SVR : {'svr__C': 1000, 'svr__epsilon': 1, 'svr__gamma': 0.01}
Modèle retenu : Polynomiale
MAE sur Test : 6.2502
RMSE sur Test : 11.3782
R² sur Test : 0.999
R² sur Train : 0.9992

========== SEED = 123 ==========
Meilleur degré polynomial : 2
Meilleurs paramètres SVR : {'svr__C': 1000, 'svr__epsilon': 1, 'svr__gamma': 0.01}
Modèle retenu : Polynomiale
MAE sur Test : 5.6852
RMSE sur Test : 1

Bilan comparatif et conclusion
Les résultats montrent que le meilleur degré polynomial reste égal à 2 pour les six seeds. Les meilleurs hyperparamètres du SVR restent également identiques : \(C=1000\), \(\epsilon=1\) et \(\gamma=0,01\). La régression polynomiale est donc retenue pour tous les découpages.
Les performances varient légèrement selon la seed : la MAE est comprise entre 5,5067 et 6,4240, la RMSE entre 10,4953 et 12,4639, et le \(R^2\) entre 0,9988 et 0,9991. Malgré ces variations, les scores restent très élevés, ce qui indique une bonne stabilité du modèle face aux découpages testés.
Il ne serait pas correct de choisir la seed donnant le meilleur résultat sur Test, car cela reviendrait à utiliser les données de test pour sélectionner le modèle ou le découpage. Cela introduirait un biais dans l'évaluation. La seed doit être fixée indépendamment des résultats obtenus sur Test.